<a href="https://colab.research.google.com/github/Daniela-Alves2004/data-mining/blob/main/Aula_FeautureSelection_Alzheimer_25_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [330]:
import pandas as pd

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.feature_selection import SelectKBest, mutual_info_classif,RFE,SelectFromModel

In [331]:
# Carregamento da base de dados
df = pd.read_csv('alzheimers_disease_data.csv')

In [332]:
# Tipos de dados
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2149 entries, 0 to 2148
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   PatientID                  2149 non-null   int64  
 1   Age                        2149 non-null   int64  
 2   Gender                     2149 non-null   int64  
 3   Ethnicity                  2149 non-null   int64  
 4   EducationLevel             2149 non-null   int64  
 5   BMI                        2149 non-null   float64
 6   Smoking                    2149 non-null   int64  
 7   AlcoholConsumption         2149 non-null   float64
 8   PhysicalActivity           2149 non-null   float64
 9   DietQuality                2149 non-null   float64
 10  SleepQuality               2149 non-null   float64
 11  FamilyHistoryAlzheimers    2149 non-null   int64  
 12  CardiovascularDisease      2149 non-null   int64  
 13  Diabetes                   2149 non-null   int64

In [333]:
df['DoctorInCharge'].head()

,DoctorInCharge
0,XXXConfid
1,XXXConfid
2,XXXConfid
3,XXXConfid
4,XXXConfid


### Pré-processamento
Após umam breve análise:
  * não é necessário aplicar substituições (NaN);
  * Não é necessário tranformações (OHE)

In [334]:
df['DoctorInCharge'].value_counts() # ver valores da coluna

,count
DoctorInCharge,
XXXConfid,2149


In [335]:
# Preparação da base
## Remoção das colunas irrelevantes: [PatientID, DoctorInCharge]

cols_remover = ['PatientID', 'DoctorInCharge']
df = df.drop(columns=cols_remover, axis=1)

In [336]:
df['Diagnosis'].value_counts()

,count
Diagnosis,
0,1389
1,760


In [337]:
# Separação de Treino e Teste
X = df.drop(columns = ['Diagnosis'], axis=1)
y = df['Diagnosis']

# Aplica a separação com train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.30,
    stratify = y
)

In [338]:
X_train.shape, X_test.shape

((1504, 32), (645, 32))

In [339]:
# Aplicar a normalização no treino e teste
## Como todas as colunas são numéricas, n
## Não é necessário filtrar com select_dtypes
numeric_cols = X_train.columns.tolist()

# Normalização
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

In [340]:
# Aplica somente o transform no test
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [341]:
# Avalia o modelo com todas as features
clf = DecisionTreeClassifier(random_state= 42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.90      0.91       417
           1       0.82      0.86      0.84       228

    accuracy                           0.89       645
   macro avg       0.87      0.88      0.88       645
weighted avg       0.89      0.89      0.89       645



In [342]:
# Aplicar a seleção de características filter
## Mutual Information

# Especificar o conjunto minimo de features
tam_f = 10

selector = SelectKBest(
    score_func = mutual_info_classif,
    k = tam_f
)

# Treinar a seleção
selector.fit(X_train, y_train)

SelectKBest(score_func=<function mutual_info_classif at 0x7a9388de09a0>)

In [343]:
selector.get_support()

array([False, False, False, False, False, False,  True, False, False,
       False, False, False, False,  True,  True, False,  True, False,
       False, False, False, False,  True,  True,  True,  True,  True,
       False, False, False,  True, False])

In [344]:
# Mostrar as features
features_filter = X_train.columns[selector.get_support()].tolist()

In [345]:
features_filter

['AlcoholConsumption',
 'Depression',
 'HeadInjury',
 'SystolicBP',
 'MMSE',
 'FunctionalAssessment',
 'MemoryComplaints',
 'BehavioralProblems',
 'ADL',
 'DifficultyCompletingTasks']

In [346]:
X_train[features_filter]

,AlcoholConsumption,Depression,HeadInjury,SystolicBP,MMSE,FunctionalAssessment,MemoryComplaints,BehavioralProblems,ADL,DifficultyCompletingTasks
2046,-0.321013,-0.510576,-0.325396,0.945939,-1.339644,1.295692,-0.507469,2.261376,-1.463262,-0.427071
234,0.005182,-0.510576,-0.325396,0.637864,-1.086902,-0.037501,-0.507469,-0.442209,1.367908,2.341529
1420,-0.961610,-0.510576,-0.325396,0.021713,0.944786,1.154488,-0.507469,2.261376,1.161823,2.341529
905,-1.526322,1.958574,-0.325396,-0.363382,-1.470927,-0.017500,-0.507469,-0.442209,1.656318,-0.427071
702,1.403157,-0.510576,3.073181,1.138487,1.378046,-1.658506,-0.507469,-0.442209,-1.051177,-0.427071
...,...,...,...,...,...,...,...,...,...,...
1482,0.330141,-0.510576,-0.325396,1.446562,-1.069965,-0.123601,-0.507469,-0.442209,-0.462039,-0.427071
306,-1.065236,-0.510576,-0.325396,-0.632948,0.541842,-0.766463,-0.507469,-0.442209,0.674520,-0.427071
140,-1.594535,-0.510576,-0.325396,-0.363382,-0.636461,-1.084835,-0.507469,-0.442209,-0.825952,-0.427071
579,0.505978,-0.510576,-0.325396,-0.016797,-0.810338,-0.680455,-0.507469,-0.442209,-0.475336,-0.427071


In [347]:
# Re-treinar o modelo com as selected_features
clf_filter = DecisionTreeClassifier(random_state = 42)
clf_filter.fit(X_train[features_filter], y_train)
y_pred_filter = clf_filter.predict(X_test[features_filter])

In [348]:
# Avalia o modelo com as features selecionadas
print(classification_report(y_test, y_pred_filter ))

              precision    recall  f1-score   support

           0       0.92      0.90      0.91       417
           1       0.83      0.86      0.85       228

    accuracy                           0.89       645
   macro avg       0.88      0.88      0.88       645
weighted avg       0.89      0.89      0.89       645



In [349]:
# Avalia o modelo com as features selecionadas
print(classification_report(y_test, y_pred ))

              precision    recall  f1-score   support

           0       0.92      0.90      0.91       417
           1       0.82      0.86      0.84       228

    accuracy                           0.89       645
   macro avg       0.87      0.88      0.88       645
weighted avg       0.89      0.89      0.89       645



* Qual o subconjunto mínimo para k (SelectKBest)?
* Mutual Information é a melhor abordagem?

##Abordagem Wrapper

Tecnica RFE(ELIMINACAOI RECURSSIVA)
Modelo DecisionThreeclassifier

In [350]:
clt_wrapper = DecisionTreeClassifier(random_state=42)

#RFE
wrapper_selection = RFE(
    estimator = clt_wrapper,
    n_features_to_select = tam_f,
)

wrapper_selection.fit(X_train, y_train)

RFE(estimator=DecisionTreeClassifier(random_state=42), n_features_to_select=10)

In [351]:
#SDelectionar melhores features
features_wrapper = X_train.columns[wrapper_selection.get_support()].tolist()

In [352]:
features_wrapper

['PhysicalActivity',
 'DietQuality',
 'CholesterolLDL',
 'CholesterolHDL',
 'CholesterolTriglycerides',
 'MMSE',
 'FunctionalAssessment',
 'MemoryComplaints',
 'BehavioralProblems',
 'ADL']

In [353]:
features_filter

['AlcoholConsumption',
 'Depression',
 'HeadInjury',
 'SystolicBP',
 'MMSE',
 'FunctionalAssessment',
 'MemoryComplaints',
 'BehavioralProblems',
 'ADL',
 'DifficultyCompletingTasks']

Embedded
selectFrommodel
decisionthreeClassifier

In [354]:
import numpy as np

clf_embedded = DecisionTreeClassifier(random_state=42)

embedded_selection = SelectFromModel(
    estimator = clf_embedded,
    threshold=-np.inf,
    max_features = tam_f
)

embedded_selection.fit(X_train, y_train)

SelectFromModel(estimator=DecisionTreeClassifier(random_state=42),
                max_features=10, threshold=-inf)

In [355]:
features_embedded = X_train.columns[embedded_selection.get_support()].tolist()

In [356]:
features_embedded

['AlcoholConsumption',
 'DietQuality',
 'CholesterolLDL',
 'CholesterolHDL',
 'CholesterolTriglycerides',
 'MMSE',
 'FunctionalAssessment',
 'MemoryComplaints',
 'BehavioralProblems',
 'ADL']

Comparar 3 abrodagens

In [357]:
def avaliar_modelos (features):
  clf = DecisionTreeClassifier(random_state=42)
  clf.fit(X_train[features], y_train)
  pred = clf.predict(X_test[features])
  return pred

In [358]:
print("Abordagem Filter")
print(classification_report(y_test, avaliar_modelos(features_filter)))

Abordagem Filter
              precision    recall  f1-score   support

           0       0.92      0.90      0.91       417
           1       0.83      0.86      0.85       228

    accuracy                           0.89       645
   macro avg       0.88      0.88      0.88       645
weighted avg       0.89      0.89      0.89       645



In [359]:
print("Abordagem wrapper")
print(classification_report(y_test, avaliar_modelos(features_wrapper)))

Abordagem wrapper
              precision    recall  f1-score   support

           0       0.92      0.92      0.92       417
           1       0.85      0.85      0.85       228

    accuracy                           0.90       645
   macro avg       0.89      0.89      0.89       645
weighted avg       0.90      0.90      0.90       645



In [360]:
print("Abordagem Embedded")
print(classification_report(y_test, avaliar_modelos(features_embedded)))

Abordagem Embedded
              precision    recall  f1-score   support

           0       0.93      0.91      0.92       417
           1       0.84      0.88      0.86       228

    accuracy                           0.90       645
   macro avg       0.89      0.90      0.89       645
weighted avg       0.90      0.90      0.90       645



In [361]:
set_filter = set(features_filter)
set_wrapper = set(features_wrapper)
set_embedded = set(features_embedded)

In [362]:
#features presentes nas tres asboradagens
interst_f = set_filter.intersection(set_wrapper).intersection(set_embedded)

In [363]:
interst_f

{'ADL',
 'BehavioralProblems',
 'FunctionalAssessment',
 'MMSE',
 'MemoryComplaints'}

In [364]:
features_comum  = list(interst_f)

In [365]:
features_comum

['ADL',
 'FunctionalAssessment',
 'MMSE',
 'MemoryComplaints',
 'BehavioralProblems']

Avaliar o modelo apenas com as features simelares

In [366]:
print("features em comum")
print(classification_report(y_test, avaliar_modelos(features_comum)))

features em comum
              precision    recall  f1-score   support

           0       0.92      0.93      0.92       417
           1       0.87      0.85      0.86       228

    accuracy                           0.90       645
   macro avg       0.89      0.89      0.89       645
weighted avg       0.90      0.90      0.90       645

